# 01b — Train SDXL Character LoRA (ai-toolkit)

Trains an SDXL character LoRA from your captioned reference images.

**Runtime:** A100 (40 GB) recommended. Training 2000 steps with rank 16 takes ~20–40 min on A100.
T4 (16 GB) works for smaller rank / lower batch but is tight and slower.

**Prerequisites:** Run `01a_caption_refs.ipynb` first to prepare images + captions.

**Output:** `<DRIVE_BASE>/loras/<CHARACTER_NAME>_sdxl.safetensors`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── Config — change these ────────────────────────────────────────────────
CHARACTER_NAME = 'MarcD'
TRIGGER_TOKEN  = 'sks_marcd'
TRAIN_STEPS    = 2250    # 1500–3000; bump to 2500 if identity is weak
LORA_RANK      = 32      # 32 for more identity detail (needs ~16 GB VRAM)
LEARNING_RATE  = '1.0e-4'
# ─────────────────────────────────────────────────────────────────────────

DRIVE_BASE  = '/content/drive/MyDrive/ai_character_studio'
CHAR_DIR    = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}'
REF_DIR     = f'{CHAR_DIR}/reference-images'
LORAS_DIR   = f'{DRIVE_BASE}/loras'
OUTPUT_LORA = f'{LORAS_DIR}/{CHARACTER_NAME}_sdxl.safetensors'

import os
os.makedirs(LORAS_DIR, exist_ok=True)

refs = [f for f in os.listdir(REF_DIR) if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))]
print(f'Character: {CHARACTER_NAME} | Trigger: {TRIGGER_TOKEN}')
print(f'Reference images: {len(refs)}')
if len(refs) < 10:
    print('WARNING: fewer than 10 images may cause weak identity learning.')

In [ ]:
# Pin a known-good stable combo to escape Colab's version churn.
# diffusers 0.31.0 + transformers 4.46.3 is a battle-tested pair for SDXL DreamBooth LoRA.
# 0.31.0 does NOT have autoencoder_rae (which needs Dinov2WithRegistersConfig from newer transformers).

# Fix broken system scipy first
!pip install -q "scipy>=1.15.0"

# Pin the compatible trio (install together so pip resolves consistently)
!pip install -q "diffusers==0.31.0" "transformers==4.46.3" "huggingface_hub<0.26" "peft==0.13.2"
!pip install -q accelerate bitsandbytes

# Get the training script FROM the matching version tag (not main — main needs newer diffusers)
EXAMPLES_DIR = '/content/diffusers_examples'
import os, shutil
if os.path.exists(EXAMPLES_DIR):
    shutil.rmtree(EXAMPLES_DIR)
!git clone --depth 1 --branch v0.31.0 https://github.com/huggingface/diffusers.git {EXAMPLES_DIR}
TRAIN_SCRIPT = f'{EXAMPLES_DIR}/examples/dreambooth/train_dreambooth_lora_sdxl.py'

assert os.path.exists(TRAIN_SCRIPT), f"Script not found: {TRAIN_SCRIPT}"
print(f'Training script (v0.31.0): {TRAIN_SCRIPT}')

# Verify the pinned versions import cleanly in a subprocess
import subprocess, sys
check = subprocess.run([sys.executable, '-c',
    'import diffusers, transformers; '
    'print("diffusers:", diffusers.__version__); '
    'print("transformers:", transformers.__version__)'],
    capture_output=True, text=True)
print(check.stdout.strip())
if check.returncode != 0:
    print('IMPORT CHECK FAILED:'); print(check.stderr[-500:])
else:
    print('✅ diffusers + transformers import cleanly')

In [ ]:
# Check GPU
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f'CUDA: {torch.cuda.is_available()}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

In [ ]:
import os, json, shutil

# The diffusers DreamBooth script expects ONLY images in instance_data_dir.
# Our REF_DIR also has .txt caption sidecars (from 01a) which crash it.
# Copy just the images into a clean training folder.
INSTANCE_DIR = '/content/instance_images'
if os.path.exists(INSTANCE_DIR):
    shutil.rmtree(INSTANCE_DIR)
os.makedirs(INSTANCE_DIR)

img_exts = ('.jpg', '.jpeg', '.png', '.webp')
N_IMAGES = 0
for f in os.listdir(REF_DIR):
    if f.lower().endswith(img_exts):
        shutil.copy2(f'{REF_DIR}/{f}', f'{INSTANCE_DIR}/{f}')
        N_IMAGES += 1

# Accelerate config — bf16 on A100
ACCEL_CONFIG = '/content/accelerate_config.yaml'
with open(ACCEL_CONFIG, 'w') as f:
    f.write("""compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
mixed_precision: 'bf16'
num_machines: 1
num_processes: 1
""")

VALIDATION_PROMPT = f'{TRIGGER_TOKEN}, portrait photo, detailed face, dramatic lighting'

print(f'Training config:')
print(f'  Character: {CHARACTER_NAME} | Trigger: {TRIGGER_TOKEN}')
print(f'  Clean instance dir: {INSTANCE_DIR} ({N_IMAGES} images, no .txt files)')
print(f'  Steps: {TRAIN_STEPS} | Rank: {LORA_RANK} | LR: {LEARNING_RATE}')
print(f'  Validation prompt: {VALIDATION_PROMPT}')

In [ ]:
import time, os

TRAIN_OUTPUT_DIR = '/content/training_output'
os.makedirs(TRAIN_OUTPUT_DIR, exist_ok=True)

# Uses INSTANCE_DIR (clean images only, no .txt) from Cell b4
start = time.time()
!accelerate launch --num_processes=1 --num_machines=1 --mixed_precision=bf16 --dynamo_backend=no \
    {TRAIN_SCRIPT} \
    --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
    --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
    --instance_data_dir="{INSTANCE_DIR}" \
    --instance_prompt="{TRIGGER_TOKEN}" \
    --output_dir="{TRAIN_OUTPUT_DIR}" \
    --mixed_precision="bf16" \
    --resolution=1024 \
    --train_batch_size=1 \
    --gradient_accumulation_steps=1 \
    --learning_rate={LEARNING_RATE} \
    --lr_scheduler="constant" \
    --lr_warmup_steps=0 \
    --max_train_steps={TRAIN_STEPS} \
    --rank={LORA_RANK} \
    --validation_prompt="{VALIDATION_PROMPT}" \
    --validation_epochs=25 \
    --seed=42 \
    --checkpointing_steps=500 \
    --gradient_checkpointing \
    --use_8bit_adam

elapsed = time.time() - start
print(f'\nTraining finished in {elapsed/60:.1f} minutes.')

In [ ]:
# Copy the trained LoRA to Drive
import glob, shutil, os

# diffusers script saves as pytorch_lora_weights.safetensors in output_dir
candidates = glob.glob(f'{TRAIN_OUTPUT_DIR}/*.safetensors')
if not candidates:
    candidates = glob.glob(f'{TRAIN_OUTPUT_DIR}/**/*.safetensors', recursive=True)

if candidates:
    # Prefer the final pytorch_lora_weights.safetensors over checkpoint copies
    final = [c for c in candidates if 'pytorch_lora_weights' in c and 'checkpoint' not in c]
    latest = final[0] if final else max(candidates, key=os.path.getmtime)
    shutil.copy2(latest, OUTPUT_LORA)
    print(f'✅ LoRA saved to Drive: {OUTPUT_LORA}')
    print(f'   Source: {latest}')
    print(f'   Size: {os.path.getsize(OUTPUT_LORA)/1024**2:.1f} MB')
else:
    print('ERROR: No .safetensors found. Check training log above.')
    print('Files in output dir:', os.listdir(TRAIN_OUTPUT_DIR))

In [ ]:
# Update metadata.json on Drive
import json
meta_path = f'{CHAR_DIR}/metadata.json'
metadata = {}
if os.path.exists(meta_path):
    with open(meta_path) as f:
        metadata = json.load(f)

metadata.update({
    'name': CHARACTER_NAME,
    'trigger': TRIGGER_TOKEN,
    'base_model': 'sdxl',
    'lora_path': OUTPUT_LORA,
    'train_steps': TRAIN_STEPS,
    'lora_rank': LORA_RANK,
})
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'metadata.json updated: {meta_path}')
print(json.dumps(metadata, indent=2))

In [ ]:
# Quick validation — generate 2 sample images using diffusers pipeline
# (This is just a fast check; full validation is done via ComfyUI in 02_test_stills.ipynb)
from diffusers import DiffusionPipeline, AutoencoderKL
import torch
from PIL import Image

vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=torch.float16)
pipe = DiffusionPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    vae=vae, torch_dtype=torch.float16, variant='fp16'
).to('cuda')
pipe.load_lora_weights(OUTPUT_LORA)

prompts = [
    f'{TRIGGER_TOKEN}, portrait, detailed face, dramatic lighting',
    f'{TRIGGER_TOKEN}, full body, standing, outdoor scene',
]
images = pipe(prompts, num_inference_steps=30, guidance_scale=7.0).images

sample_dir = f'{CHAR_DIR}/samples'
os.makedirs(sample_dir, exist_ok=True)
for i, img in enumerate(images):
    path = f'{sample_dir}/validation_{i:02d}.png'
    img.save(path)
    print(f'Sample saved: {path}')

# Display
from IPython.display import display
for img in images:
    display(img.resize((512, 512)))

print('\n✅ Training complete. Check the samples above vs your reference images.')
print('If face is drifting: bump TRAIN_STEPS to 2500 or LORA_RANK to 32 and retrain.')
print('Next: run 02_test_stills.ipynb for full ComfyUI stills generation.')